In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import hamming_loss, accuracy_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Concatenate
from tensorflow.keras.models import Model
from sklearn.model_selection import ParameterGrid

In [3]:
data2=pd.read_excel('final33.xlsx')

In [4]:
specified_column = data2['3-2.'].explode().tolist()  # Flatten the list of lists

In [5]:
# convert to list
numbers = [int(num) if isinstance(num, (int, float)) else [int(x) for x in num.split(',')] for num in specified_column]
numbers = [num if isinstance(num, list) else [num] for num in numbers]

In [14]:
for col_val, num_list in zip(specified_column, numbers):
    print('each number %s is %s' % (col_val, num_list))

each number 4,7,8,10 is [4, 7, 8, 10]
each number 1,3,4,7,9 is [1, 3, 4, 7, 9]
each number 1,2,3,5,6,9 is [1, 2, 3, 5, 6, 9]
each number 1,2,7,11,12 is [1, 2, 7, 11, 12]
each number 1,2,6,11 is [1, 2, 6, 11]
each number 1,2,3,5,12 is [1, 2, 3, 5, 12]
each number 3,6,7,11 is [3, 6, 7, 11]
each number 1,5 is [1, 5]
each number 9 is [9]
each number 11 is [11]
each number 1,2,3,5,11,12 is [1, 2, 3, 5, 11, 12]
each number 5,1 is [5, 1]
each number 2,10 is [2, 10]
each number 1,5,11 is [1, 5, 11]
each number 1,2,3,5,6,11,12 is [1, 2, 3, 5, 6, 11, 12]
each number 1,2,3,9,10 is [1, 2, 3, 9, 10]
each number 3,4,5,10,11 is [3, 4, 5, 10, 11]
each number 1,2,3,6,8,11,12 is [1, 2, 3, 6, 8, 11, 12]
each number 2,4,10,11 is [2, 4, 10, 11]
each number 1,2,6,7,10,11 is [1, 2, 6, 7, 10, 11]
each number 2,4,5,6,9,11 is [2, 4, 5, 6, 9, 11]
each number 1,2,3,4,5,9 is [1, 2, 3, 4, 5, 9]
each number 1,11 is [1, 11]
each number 1,5,12 is [1, 5, 12]
each number 1,6,8,10 is [1, 6, 8, 10]
each number 10,7,6 is [

In [ ]:
total=pd.read_csv('/content/total.csv')

In [ ]:
total['user_pref']=numbers
total=total.drop(columns=['kind_adv'])

In [ ]:
new_data = []
for i in range(len(total)):
    row_data = total.iloc[i].to_dict()
    for j in total['user_pref'][i]:
        new_row = row_data.copy()
        new_row['user_pref'] = j
        new_data.append(new_row)

new_df = pd.DataFrame(new_data)
# print(new_df)

In [ ]:
null_records =new_df[new_df.isnull().any(axis=1)]
print(null_records)

Empty DataFrame
Columns: [adv, dist_code, line, trip_purpose, siting, transfer, age, totall_ travel_ time, male, single, degree, semi_employ_3_4, full_employ_5_6_7, no_job_1_2, income, peak, number _trips, adv_place(0=no,1=adv_in,2=adv_out,3=both), user_pref]
Index: []


In [ ]:
new_df.to_excel('/content/drive/MyDrive/thesis/data_decisionTree.xlsx')

# ***part2:***

# new categories:

In [ ]:
df=pd.read_excel('/content/drive/MyDrive/thesis/data_decisionTree.xlsx')

In [ ]:
df.loc[df['user_pref'] == 2, 'user_pref'] = 1
df.loc[df['user_pref'] == 11, 'user_pref'] = 2
df.loc[df['user_pref'] == 12, 'user_pref'] = 2
df.loc[df['user_pref'] == 5, 'user_pref'] = 3
df.loc[df['user_pref'] == 6, 'user_pref'] = 4
df.loc[df['user_pref'] == 7, 'user_pref'] = 5
df.loc[df['user_pref'] == 8, 'user_pref'] = 5
df.loc[df['user_pref'] == 9, 'user_pref'] = 6
df.loc[df['user_pref'] == 10, 'user_pref'] = 6

# ***models:***

# 1.decision Tree

In [ ]:
def analyze_ad_preferences_decisionTree(df, target_col='user_pref', test_size=0.1, random_state=42):
  X = df.copy()
  X = X.drop(columns=['user_pref'])
  y=df[target_col]
  scaler = StandardScaler()
  X = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
  param_grid = {
      'max_depth':[2,3,4,5,6,7]
    }

  dt = DecisionTreeClassifier( random_state=42,)
  grid_search = GridSearchCV(
        dt, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1
    )
  grid_search.fit(X_train, y_train)
  best_dt = grid_search.best_estimator_
  best_dt.fit(X_train, y_train)
  print(grid_search.best_params_ )
  best_dt_importance = pd.DataFrame({
        'feature': X.columns,
        'importance': best_dt.feature_importances_
    }).sort_values('importance', ascending=False)
  dt_pred = best_dt.predict(X_test)
  target_names = y_test.unique()
  dt_report = classification_report(y_test, dt_pred,
                                    target_names=target_names,
                                    output_dict=True)
  cv_scores_dt = cross_val_score(best_dt, X_train, y_train, cv=5)
  category_profiles = {}
  for cat in np.unique(y_train):
        cat_mask = (y_train == cat)
        profile = X_train[cat_mask].mean()
        category_profiles[cat] = profile
  return {
        'models': {
            'decision_tree': best_dt,
        },
        'importance': {
            'decision_tree': best_dt_importance,
        },
        'evaluation': {
            'decision_tree': {
                'report': dt_report,
                'cv_scores': cv_scores_dt,
                'confusion_matrix': confusion_matrix(y_test, dt_pred)
            }
        },
        'category_profiles': category_profiles
    }


In [ ]:
# Load data
df.drop(columns=['Unnamed: 0'])
# Run the analysis
results = analyze_ad_preferences_decisionTree(df, target_col='user_pref')

# Visualize the results
# plot_results(results)

# Access specific results
print("Decision Tree Performance:")
print(results['evaluation']['decision_tree']['report'])

# Access category profiles
# print("\nCategory Profiles:")
# print(results['category_profiles'])

Fitting 5 folds for each of 6 candidates, totalling 30 fits
{'max_depth': 3}
Decision Tree Performance:
{2: {'precision': 0.23809523809523808, 'recall': 0.3125, 'f1-score': 0.2702702702702703, 'support': 16.0}, 0: {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 18.0}, 5: {'precision': 0.2, 'recall': 0.5, 'f1-score': 0.2857142857142857, 'support': 14.0}, 3: {'precision': 0.14285714285714285, 'recall': 0.3, 'f1-score': 0.1935483870967742, 'support': 10.0}, 4: {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 12.0}, 1: {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 15.0}, 'accuracy': 0.17647058823529413, 'macro avg': {'precision': 0.09682539682539681, 'recall': 0.18541666666666667, 'f1-score': 0.1249221571802217, 'support': 85.0}, 'weighted avg': {'precision': 0.0945658263305322, 'recall': 0.17647058823529413, 'f1-score': 0.12070362582696549, 'support': 85.0}}


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# 2.random forest

In [ ]:
def analyze_ad_preferences_randomforest(df, target_col='user_pref', test_size=0.1, random_state=42):
  X = df.copy()
  X = X.drop(columns=['user_pref'])
  y=df[target_col]
  scaler = StandardScaler()
  X = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
  param_grid = {
      'max_depth':[2,3,4,5,6,7],
      'n_estimators':[1,10,20,30,40,50,60]}

  rf = RandomForestClassifier(random_state=random_state,criterion='entropy', )
  grid_search = GridSearchCV(
        rf, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1
    )
  grid_search.fit(X_train, y_train)
  best_rf = grid_search.best_estimator_
  best_rf.fit(X_train, y_train)
  print(grid_search.best_params_ )
  rf_importance = pd.DataFrame({
        'feature': X.columns,
        'importance': best_rf.feature_importances_
    }).sort_values('importance', ascending=False)

  rf_pred = best_rf.predict(X_test)
  target_names = y_test.unique()
  rf_report = classification_report(y_test, rf_pred,
                                    target_names=target_names,
                                    output_dict=True, zero_division=0)

  cv_scores_rf = cross_val_score(best_rf, X_train, y_train, cv=5)
  category_profiles = {}
  for cat in np.unique(y_train):
        cat_mask = (y_train == cat)
        profile = X_train[cat_mask].mean()
        category_profiles[cat] = profile

  return {
        'models': {
            'random_forest': best_rf
        },
        'importance': {
            'random_forest': rf_importance
        },
        'evaluation': {
            'random_forest': {
                'report': rf_report,
                'cv_scores': cv_scores_rf,
                'confusion_matrix': confusion_matrix(y_test, rf_pred)
            }
        },
        'category_profiles': category_profiles
    }

In [ ]:

df.drop(columns=['Unnamed: 0'])
# Run the analysis
results = analyze_ad_preferences_randomforest(df, target_col='user_pref')

# Visualize the results
# plot_results(results)

print("Random Forest Performance:")
print(results['evaluation']['random_forest']['report'])

# Access category profiles
# print("\nCategory Profiles:")
# print(results['category_profiles'])

Fitting 5 folds for each of 42 candidates, totalling 210 fits
{'max_depth': 2, 'n_estimators': 30}
Random Forest Performance:
{3: {'precision': 0.1836734693877551, 'recall': 0.5625, 'f1-score': 0.27692307692307694, 'support': 16.0}, 1: {'precision': 0.17857142857142858, 'recall': 0.2777777777777778, 'f1-score': 0.21739130434782608, 'support': 18.0}, 6: {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 14.0}, 4: {'precision': 0.2, 'recall': 0.1, 'f1-score': 0.13333333333333333, 'support': 10.0}, 5: {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 12.0}, 2: {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 15.0}, 'accuracy': 0.17647058823529413, 'macro avg': {'precision': 0.09370748299319727, 'recall': 0.15671296296296297, 'f1-score': 0.10460795243403938, 'support': 85.0}, 'weighted avg': {'precision': 0.09591836734693879, 'recall': 0.17647058823529413, 'f1-score': 0.11384877696898157, 'support': 85.0}}


# 3.XGBoost

In [ ]:
df['user_pref'].value_counts()

,count
user_pref,
1,181
2,169
4,154
3,142
6,116
5,84


In [ ]:
# over-sampling
from sklearn.utils import resample

# Concatenate the original DataFrame to the minority classes after duplicating samples
df_minority3 = df[df['user_pref'] == 3]
df_minority2 = df[df['user_pref'] == 2]
df_majority = df[df['user_pref'] == 1]  # Change to majority class name
df_minority4 = df[df['user_pref'] == 4]
df_minority5 = df[df['user_pref'] == 5]
df_minority6 = df[df['user_pref'] == 6]
# Upsample minority class
df_minority_upsampled3 = resample(df_minority3,
                                  replace=True,
                                  n_samples=len(df_majority),
                                  random_state=42)
df_minority_upsampled2= resample(df_minority2,
                                  replace=True,
                                  n_samples=len(df_majority),
                                  random_state=42)
df_minority_upsampled4= resample(df_minority4,
                                  replace=True,
                                  n_samples=len(df_majority),
                                  random_state=42)
df_minority_upsampled5= resample(df_minority5,
                                  replace=True,
                                  n_samples=len(df_majority),
                                  random_state=42)
df_minority_upsampled6= resample(df_minority6,
                                  replace=True,
                                  n_samples=len(df_majority),
                                  random_state=42)


# Combine majority class with upsampled minority class
df_upsampled = pd.concat([df_majority, df_minority_upsampled3,df_minority_upsampled2,df_minority_upsampled4,df_minority_upsampled5,df_minority_upsampled6])

# Check new class distribution
print(df_upsampled['user_pref'].value_counts())

user_pref
1    181
3    181
2    181
4    181
5    181
6    181
Name: count, dtype: int64


In [ ]:
X=df_upsampled[[ 'trip_purpose',
        'age',  'male'
       ,'semi_employ_3_4', 'income', 'peak',
      'adv_place(0=no,1=adv_in,2=adv_out,3=both)',
       'user_pref']]
X['user_pref'] -=1

<ipython-input-13-4153e96d51a2>:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['user_pref'] -=1


In [ ]:
X1 = X.copy()
y = X1['user_pref']
X = X1.drop(columns=['user_pref'])
scaler = StandardScaler()
X = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)
param_grid = {
      'max_depth':[7],
      'n_estimators':[125],
      'gamma': [0],
      'reg_lambda':[ 0.1],
    'alpha': [0.1] ,
        'subsample': [0.8]}
xgb = XGBClassifier(
    objective='multi:softmax',
    random_state=42,
    learning_rate=0.1)
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42, )
grid_search = GridSearchCV(
        xgb, param_grid, cv=cv, scoring='accuracy', n_jobs=-1, verbose=1,refit=True
    )
grid_search.fit(X_train, y_train)
best_xgb = grid_search.best_estimator_
best_xgb.fit(X_train, y_train)
print(grid_search.best_params_ )
best_xgb.fit(X_train, y_train)
xgb_importance = pd.DataFrame({
        'feature': X.columns,
        'importance': best_xgb.feature_importances_
    }).sort_values('importance', ascending=False)
xgb_pred = best_xgb.predict(X_test)
target_names = y_test.unique()
xgb_report = classification_report(y_test, xgb_pred,
                                      target_names=target_names,
                                      output_dict=True,zero_division=0)
xgb_cm = confusion_matrix(y_test, xgb_pred)
xgb_cv_scores = cross_val_score(best_xgb, X_train, y_train, cv=5)

In [ ]:
explainer = shap.TreeExplainer(best_xgb,)
shap_values = explainer.shap_values(X_test,)
class_shap_importance = {}
feature_names = X.columns.tolist()
feature_signs = {}
target_names = np.unique(y)
mean_shap_values=0
abs_importance=0
for i in range(len(shap_values)):
  mean_shap_values+=shap_values[i]
  abs_importance+=np.abs(shap_values[i])
mean_shap_values/=shap_values.shape[0]
abs_importance/=shap_values.shape[0]
for class_name in range(len(target_names)):
  feature_signs[class_name] = pd.DataFrame({
            'feature': X.columns,
            'importance': abs_importance[:,class_name],
            'mean_shap_value': mean_shap_values[:,class_name],
            'sign': np.where(mean_shap_values[:,class_name] >= 0, '+', '-')
        }).sort_values('importance', ascending=False)

  feature_signs[class_name]['effect'] = feature_signs[class_name].apply(
            lambda row: f"{'increases' if row['sign'] == '+' else 'decreases'} probability of class {class_name}",
            axis=1
        )
xgb_cm = metric.confusion_matrix(y_test, xgb_pred)
xgb_cv_scores = cross_val_score(best_xgb, X_train, y_train, cv=cv)

print(f"Feature names length: {len(feature_names)}")
# print(f"SHAP values shape for class 0: {shap_values[3].shape}")

# 5.ANN+logestic

In [ ]:
def build_hybrid_model(wide_dim, deep_dim, num_classes, hidden_units=32):
    # Wide (Linear) input
    wide_input = Input(shape=(wide_dim,), name='wide_input')

    # Deep (ANN) input
    deep_input = Input(shape=(deep_dim,), name='deep_input')

    # Deep path - simple ANN with one hidden layer
    deep = Dense(hidden_units, activation='relu')(deep_input)

    # Combine wide and deep paths
    combined = Concatenate()([wide_input, deep])

    # Output layer
    if num_classes == 2:
        output = Dense(1, activation='sigmoid')(combined)
    else:
        output = Dense(num_classes+1, activation='softmax')(combined)

    # Create model
    model = Model(inputs=[wide_input, deep_input], outputs=output)

    # Compile model
    if num_classes == 2:
        model.compile(optimizer='adam',
                     loss='binary_crossentropy',
                     metrics=['accuracy'])
    else:
        model.compile(optimizer='adam',
                     loss='sparse_categorical_crossentropy',
                     metrics=['accuracy'])

    return model

def prepare_data(wide_features, deep_features, labels, test_size=0.2):
    # Scale features
    wide_scaler = StandardScaler()
    deep_scaler = StandardScaler()

    wide_scaled = wide_scaler.fit_transform(wide_features)
    deep_scaled = deep_scaler.fit_transform(deep_features)

    # Split data
    (wide_train, wide_test,
     deep_train, deep_test,
     y_train, y_test) = train_test_split(wide_scaled, deep_scaled, labels,
                                        test_size=test_size,
                                        random_state=42)

    return (wide_train, wide_test, deep_train, deep_test,
            y_train, y_test, wide_scaler, deep_scaler)

def train_hybrid_model(wide_features, deep_features, labels,
                      hidden_units=32, epochs=50, batch_size=32,
                      test_size=0.1, verbose=1):
    # Prepare data
    (wide_train, wide_test, deep_train, deep_test,
     y_train, y_test, wide_scaler, deep_scaler) = prepare_data(
        wide_features, deep_features, labels, test_size
    )

    # Get number of classes
    num_classes = len(np.unique(labels)) if len(labels.shape) == 1 else labels.shape[1]

    # Build model
    model = build_hybrid_model(
        wide_dim=wide_features.shape[1],
        deep_dim=deep_features.shape[1],
        num_classes=num_classes,
        hidden_units=hidden_units
    )

    # Train model
    history = model.fit(
        [wide_train, deep_train], y_train,
        validation_data=([wide_test, deep_test], y_test),
        epochs=epochs,
        batch_size=batch_size,
        verbose=verbose
    )

    return model, history, (wide_scaler, deep_scaler)

def predict_hybrid(model, wide_features, deep_features, scalers):

    wide_scaler, deep_scaler = scalers
    wide_scaled = wide_scaler.transform(wide_features)
    deep_scaled = deep_scaler.transform(deep_features)

    return model.predict([wide_scaled, deep_scaled])

In [ ]:
wide_features = df_upsampled[['age', 'income', 'full_employ_5_6_7','siting', 'male', 'peak']]
deep_features = df_upsampled[[ 'adv','age','adv_place(0=no,1=adv_in,2=adv_out,3=both)','trip_purpose','number _trips']]
labels = df_upsampled['user_pref']

In [ ]:
model, history, scalers = train_hybrid_model(wide_features, deep_features, labels)

Epoch 1/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.1165 - loss: 2.2369 - val_accuracy: 0.1765 - val_loss: 2.1261
Epoch 2/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.1533 - loss: 2.0858 - val_accuracy: 0.1765 - val_loss: 2.0647
Epoch 3/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.1847 - loss: 2.0166 - val_accuracy: 0.1882 - val_loss: 2.0297
Epoch 4/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.2027 - loss: 1.9364 - val_accuracy: 0.1647 - val_loss: 2.0027
Epoch 5/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.1832 - loss: 1.9232 - val_accuracy: 0.2000 - val_loss: 1.9832
Epoch 6/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.2216 - loss: 1.8734 - val_accuracy: 0.1529 - val_loss: 1.9719
Epoch 7/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2333 - loss: 1.8378 - val_accuracy: 0.1529 - val_loss: 1.9567
Epoch 8/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2222 - loss: 1.8294 - val_accuracy: 0.1529 - val_loss

In [ ]:
wide_scaler = StandardScaler()
deep_scaler = StandardScaler()
wide_scaled = wide_scaler.fit_transform(wide_features)
deep_scaled = deep_scaler.fit_transform(deep_features)
(wide_train, wide_test,
  deep_train, deep_test,
  y_train, y_test) = train_test_split(wide_scaled, deep_scaled, labels,
                                        test_size=0.1,
                                        random_state=42)
param_grid = {
            'hidden_units': [16, 32, 64],
            'epochs': [30, 50, 100],
            'batch_size': [16, 32, 64]
        }
num_classes = len(np.unique(labels)) if len(labels.shape) == 1 else labels.shape[1]
wide_dim=wide_features.shape[1]
deep_dim=deep_features.shape[1]
num_classes=num_classes
hidden_units=32
wide_input = Input(shape=(wide_dim,), name='wide_input')
deep_input = Input(shape=(deep_dim,), name='deep_input')
deep = Dense(hidden_units, activation='relu')(deep_input)
combined = Concatenate()([wide_input, deep])
output = Dense(num_classes+1, activation='softmax')(combined)
model = Model(inputs=[wide_input, deep_input], outputs=output)
model.compile(optimizer='adam',
                     loss='sparse_categorical_crossentropy',
                     metrics=['accuracy'])

In [ ]:
def build_hybrid_model(wide_dim, deep_dim, num_classes, hidden_units=32):
    # Wide (Linear) input
    wide_input = Input(shape=(wide_dim,), name='wide_input')

    # Deep (ANN) input
    deep_input = Input(shape=(deep_dim,), name='deep_input')

    # Deep path - simple ANN with one hidden layer
    deep = Dense(hidden_units, activation='relu')(deep_input)

    # Combine wide and deep paths
    combined = Concatenate()([wide_input, deep])

    # Output layer
    if num_classes == 2:
        output = Dense(1, activation='sigmoid')(combined)
    else:
        output = Dense(num_classes+1, activation='softmax')(combined)

    # Create model
    model = Model(inputs=[wide_input, deep_input], outputs=output)

    # Compile model
    if num_classes == 2:
        model.compile(optimizer='adam',
                     loss='binary_crossentropy',
                     metrics=['accuracy'])
    else:
        model.compile(optimizer='adam',
                     loss='sparse_categorical_crossentropy',
                     metrics=['accuracy'])

    return model

def prepare_data(wide_features, deep_features, labels, test_size=0.1):
    # Scale features
    wide_scaler = StandardScaler()
    deep_scaler = StandardScaler()

    wide_scaled = wide_scaler.fit_transform(wide_features)
    deep_scaled = deep_scaler.fit_transform(deep_features)

    # Split data
    (wide_train, wide_test,
     deep_train, deep_test,
     y_train, y_test) = train_test_split(wide_scaled, deep_scaled, labels,
                                        test_size=test_size,
                                        random_state=42)

    return (wide_train, wide_test, deep_train, deep_test,
            y_train, y_test, wide_scaler, deep_scaler)

def train_hybrid_model_with_gridsearch(wide_features, deep_features, labels,
                                     param_grid=None, test_size=0.1, verbose=1):

    # Default parameter grid if none provided
    if param_grid is None:
        param_grid = {
            'hidden_units': [16, 32, 64],
            'epochs': [30, 50,70, 100],
            'batch_size': [16, 32, 64]
        }

    # Prepare data
    (wide_train, wide_val, deep_train, deep_val,
     y_train, y_val, wide_scaler, deep_scaler) = prepare_data(
        wide_features, deep_features, labels, test_size
    )

    # Get number of classes
    num_classes = len(np.unique(labels)) if len(labels.shape) == 1 else labels.shape[1]

    # Initialize tracking variables
    best_score = -np.inf
    best_params = None
    best_model = None
    best_history = None
    results = []

    # Generate parameter combinations
    param_combinations = list(ParameterGrid(param_grid))

    if verbose:
        print(f"Testing {len(param_combinations)} parameter combinations...")

    # Try each parameter combination
    for params in param_combinations:
        if verbose:
            print(f"\nTesting parameters: {params}")

        # Build model with current parameters
        model = build_hybrid_model(
            wide_dim=wide_features.shape[1],
            deep_dim=deep_features.shape[1],
            num_classes=num_classes,
            hidden_units=params['hidden_units']
        )

        # Train model
        history = model.fit(
            [wide_train, deep_train], y_train,
            validation_data=([wide_val, deep_val], y_val),
            epochs=params['epochs'],
            batch_size=params['batch_size'],
            verbose=0 if verbose==1 else verbose
        )

        # Get validation score
        val_score = max(history.history['val_accuracy'])

        # Store results
        results.append({
            'params': params,
            'val_score': val_score
        })

        if verbose:
            print(f"Validation accuracy: {val_score:.4f}")

        # Update best parameters if necessary
        if val_score > best_score:
            best_score = val_score
            best_params = params
            best_model = model
            best_history = history

    if verbose:
        print("\nGrid search completed.")
        print(f"Best parameters: {best_params}")
        print(f"Best validation accuracy: {best_score:.4f}")

    return best_model, best_history, (wide_scaler, deep_scaler), {
        'best_params': best_params,
        'best_score': best_score,
        'results': results
    }

def predict_hybrid(model, wide_features, deep_features, scalers):
    wide_scaler, deep_scaler = scalers
    wide_scaled = wide_scaler.transform(wide_features)
    deep_scaled = deep_scaler.transform(deep_features)

    return model.predict([wide_scaled, deep_scaled])



In [ ]:
model, history, scalers, search_results = train_hybrid_model_with_gridsearch(
    wide_features,
    deep_features,
    labels
)

Testing 36 parameter combinations...

Testing parameters: {'batch_size': 16, 'epochs': 30, 'hidden_units': 16}
Validation accuracy: 0.2844

Testing parameters: {'batch_size': 16, 'epochs': 30, 'hidden_units': 32}
Validation accuracy: 0.2936

Testing parameters: {'batch_size': 16, 'epochs': 30, 'hidden_units': 64}
Validation accuracy: 0.3119

Testing parameters: {'batch_size': 16, 'epochs': 50, 'hidden_units': 16}
Validation accuracy: 0.3119

Testing parameters: {'batch_size': 16, 'epochs': 50, 'hidden_units': 32}
Validation accuracy: 0.2844

Testing parameters: {'batch_size': 16, 'epochs': 50, 'hidden_units': 64}
Validation accuracy: 0.2936

Testing parameters: {'batch_size': 16, 'epochs': 70, 'hidden_units': 16}
Validation accuracy: 0.2477

Testing parameters: {'batch_size': 16, 'epochs': 70, 'hidden_units': 32}
Validation accuracy: 0.3119

Testing parameters: {'batch_size': 16, 'epochs': 70, 'hidden_units': 64}
Validation accuracy: 0.3394

Testing parameters: {'batch_size': 16, 'epoc

In [ ]:
print(search_results)

{'best_params': {'batch_size': 64, 'epochs': 100, 'hidden_units': 64}, 'best_score': 0.3486238420009613, 'results': [{'params': {'batch_size': 16, 'epochs': 30, 'hidden_units': 16}, 'val_score': 0.2844036817550659}, {'params': {'batch_size': 16, 'epochs': 30, 'hidden_units': 32}, 'val_score': 0.29357796907424927}, {'params': {'batch_size': 16, 'epochs': 30, 'hidden_units': 64}, 'val_score': 0.31192660331726074}, {'params': {'batch_size': 16, 'epochs': 50, 'hidden_units': 16}, 'val_score': 0.31192660331726074}, {'params': {'batch_size': 16, 'epochs': 50, 'hidden_units': 32}, 'val_score': 0.2844036817550659}, {'params': {'batch_size': 16, 'epochs': 50, 'hidden_units': 64}, 'val_score': 0.29357796907424927}, {'params': {'batch_size': 16, 'epochs': 70, 'hidden_units': 16}, 'val_score': 0.24770642817020416}, {'params': {'batch_size': 16, 'epochs': 70, 'hidden_units': 32}, 'val_score': 0.31192660331726074}, {'params': {'batch_size': 16, 'epochs': 70, 'hidden_units': 64}, 'val_score': 0.33944

# 6.logestic

In [ ]:
def prepare_data(features, target, test_size=0.2, random_state=42):
    # Scale features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(features)

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, target,
        test_size=test_size,
        random_state=random_state,
        stratify=target
    )

    return X_train, X_test, y_train, y_test, scaler

def train_logistic_model(X_train, y_train, multi_class='auto', max_iter=1000):
    model = LogisticRegression(
        multi_class=multi_class,
        max_iter=max_iter,
        random_state=42
    )
    model.fit(X_train, y_train)
    return model

def analyze_feature_importance(model, feature_names):
    # Get coefficients
    if len(model.classes_) == 2:
        coefficients = model.coef_[0]
        classes = ['Class 0', 'Class 1']
    else:
        coefficients = model.coef_
        classes = [f'Class {i}' for i in range(len(model.classes_))]

    # Create feature importance DataFrame
    importance_df = pd.DataFrame()

    for idx, class_name in enumerate(classes):
        if len(model.classes_) == 2:
            coef = coefficients
        else:
            coef = coefficients[idx]

        class_importance = pd.DataFrame({
            'Feature': feature_names,
            'Importance': np.abs(coef),
            'Effect': coef,
            'Class': class_name
        })
        importance_df = pd.concat([importance_df, class_importance])

    return importance_df

def plot_feature_importance(importance_df, top_n=10):
    plt.figure(figsize=(12, 6))

    # Get top features by average absolute importance
    top_features = (importance_df.groupby('Feature')['Importance']
                   .mean()
                   .sort_values(ascending=False)
                   .head(top_n)
                   .index)

    # Filter data for top features
    plot_data = importance_df[importance_df['Feature'].isin(top_features)]

    # Create bar plot
    sns.barplot(data=plot_data,
                x='Effect',
                y='Feature',
                hue='Class',
                dodge=True)

    plt.title(f'Top {top_n} Most Important Features by Class')
    plt.xlabel('Coefficient Value (Effect on Prediction)')
    plt.tight_layout()
    plt.show()

def evaluate_model(model, X_test, y_test):
    # Make predictions
    y_pred = model.predict(X_test)

    # Calculate metrics
    report = classification_report(y_test, y_pred, output_dict=True)
    conf_matrix = confusion_matrix(y_test, y_pred)
    return report

def predict_new_data(model, features, scaler):
    # Scale features
    X_scaled = scaler.transform(features)

    # Get predictions and probabilities
    predictions = model.predict(X_scaled)
    probabilities = model.predict_proba(X_scaled)

    return predictions, probabilities

def run_analysis(features, target):
    # Prepare data
    X_train, X_test, y_train, y_test, scaler = prepare_data(features, target)

    # Train model
    model = train_logistic_model(X_train, y_train)

    # Analyze feature importance
    importance_df = analyze_feature_importance(model, features.columns)

    # Plot feature importance
    # plot_feature_importance(importance_df)

    # Evaluate model
    evaluation = evaluate_model(model, X_test, y_test)

    return model, scaler, importance_df, evaluation


In [ ]:
features = df[['age', 'income', 'degree','full_employ_5_6_7','siting', 'male', 'single',
               'peak','totall_ travel_ time','adv','age']]
target =df['user_pref']

model, scaler, importance_df, evaluation = run_analysis(features, target)

/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


In [ ]:
print(evaluation)

{'0': {'precision': 0.2714285714285714, 'recall': 0.5277777777777778, 'f1-score': 0.3584905660377358, 'support': 36.0}, '1': {'precision': 0.2, 'recall': 0.11764705882352941, 'f1-score': 0.14814814814814814, 'support': 34.0}, '2': {'precision': 0.21875, 'recall': 0.2413793103448276, 'f1-score': 0.22950819672131148, 'support': 29.0}, '3': {'precision': 0.3333333333333333, 'recall': 0.41935483870967744, 'f1-score': 0.37142857142857144, 'support': 31.0}, '4': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 17.0}, '5': {'precision': 0.16666666666666666, 'recall': 0.043478260869565216, 'f1-score': 0.06896551724137931, 'support': 23.0}, 'accuracy': 0.25882352941176473, 'macro avg': {'precision': 0.19836309523809526, 'recall': 0.2249395410875629, 'f1-score': 0.19609016659619102, 'support': 170.0}, 'weighted avg': {'precision': 0.2181285014005602, 'recall': 0.25882352941176473, 'f1-score': 0.22175839841265288, 'support': 170.0}}


# ***new_category2:***

In [ ]:
df=pd.read_excel('/content/drive/MyDrive/thesis/data_decisionTree.xlsx')

In [ ]:
df.loc[df['user_pref'] == 2, 'user_pref'] = 1
df.loc[df['user_pref'] == 3, 'user_pref'] = 1
df.loc[df['user_pref'] == 4, 'user_pref'] = 1
df.loc[df['user_pref'] == 5, 'user_pref'] = 2
df.loc[df['user_pref'] == 6, 'user_pref'] = 2
df.loc[df['user_pref'] == 7, 'user_pref'] = 2
df.loc[df['user_pref'] == 8, 'user_pref'] = 2
df.loc[df['user_pref'] == 9, 'user_pref'] = 3
df.loc[df['user_pref'] == 10, 'user_pref'] = 3
df.loc[df['user_pref'] == 11, 'user_pref'] = 3
df.loc[df['user_pref'] == 12, 'user_pref'] = 3

# 1.decision tree

In [ ]:
def analyze_ad_preferences_decisionTree(df, target_col='user_pref', test_size=0.1, random_state=42):
  X = df.copy()
  X = X.drop(columns=['user_pref'])
  y=df[target_col]
  scaler = StandardScaler()
  X = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
  param_grid = {
      'max_depth':[2,3,4,5,6]
    }

  dt = DecisionTreeClassifier( random_state=42,)
  grid_search = GridSearchCV(
        dt, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1
    )
  grid_search.fit(X_train, y_train)
  best_dt = grid_search.best_estimator_
  best_dt.fit(X_train, y_train)
  print(grid_search.best_params_ )
  best_dt_importance = pd.DataFrame({
        'feature': X.columns,
        'importance': best_dt.feature_importances_
    }).sort_values('importance', ascending=False)
  dt_pred = best_dt.predict(X_test)
  target_names = y_test.unique()
  dt_report = classification_report(y_test, dt_pred,
                                    target_names=target_names,
                                    output_dict=True)
  cv_scores_dt = cross_val_score(best_dt, X_train, y_train, cv=5)
  category_profiles = {}
  for cat in np.unique(y_train):
        cat_mask = (y_train == cat)
        profile = X_train[cat_mask].mean()
        category_profiles[cat] = profile
  return {
        'models': {
            'decision_tree': best_dt,
        },
        'importance': {
            'decision_tree': best_dt_importance,
        },
        'evaluation': {
            'decision_tree': {
                'report': dt_report,
                'cv_scores': cv_scores_dt,
                'confusion_matrix': confusion_matrix(y_test, dt_pred)
            }
        },
        'category_profiles': category_profiles
    }


In [ ]:
df.drop(columns=['Unnamed: 0'])
# Run the analysis
results = analyze_ad_preferences_decisionTree(df, target_col='user_pref')

# Visualize the results
# plot_results(results)

# Access specific results
print("Decision Tree Performance:")
print(results['evaluation']['decision_tree']['report'])

Fitting 5 folds for each of 5 candidates, totalling 25 fits
{'max_depth': 2}
Decision Tree Performance:
{1: {'precision': 0.4025974025974026, 'recall': 0.9393939393939394, 'f1-score': 0.5636363636363636, 'support': 33.0}, 3: {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 19.0}, 2: {'precision': 0.375, 'recall': 0.09090909090909091, 'f1-score': 0.14634146341463414, 'support': 33.0}, 'accuracy': 0.4, 'macro avg': {'precision': 0.2591991341991342, 'recall': 0.3434343434343434, 'f1-score': 0.23665927568366593, 'support': 85.0}, 'weighted avg': {'precision': 0.301890756302521, 'recall': 0.4, 'f1-score': 0.275638450502152, 'support': 85.0}}


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# 2.random forest

In [ ]:
def analyze_ad_preferences_randomforest(df, target_col='user_pref', test_size=0.1, random_state=42):
  X = df.copy()
  X = X.drop(columns=['user_pref'])
  y=df[target_col]
  scaler = StandardScaler()
  X = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
  param_grid = {
      'max_depth':[2,3,4,5,6,7],
      'n_estimators': [10, 20, 30, 40, 50, 60],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4] }

  rf = RandomForestClassifier(random_state=random_state )
  grid_search = GridSearchCV(
        rf, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1,return_train_score=True
    )
  grid_search.fit(X_train, y_train)
  best_rf = grid_search.best_estimator_
  best_rf.fit(X_train, y_train)
  print("Best parameters:", grid_search.best_params_)
  print("Best cross-validation score:", grid_search.best_score_)
  rf_importance = pd.DataFrame({
        'feature': X.columns,
        'importance': best_rf.feature_importances_
    }).sort_values('importance', ascending=False)

  rf_pred = best_rf.predict(X_test)
  target_names = y_test.unique()
  rf_report = classification_report(y_test, rf_pred,
                                    target_names=target_names,
                                    output_dict=True)
  cv_scores_rf = cross_val_score(best_rf, X_train, y_train, cv=5)
  category_profiles = {}
  for cat in np.unique(y_train):
        cat_mask = (y_train == cat)
        profile = X_train[cat_mask].mean()
        category_profiles[cat] = profile

  return {
        'models': {
            'random_forest': best_rf,
            'grid_search': grid_search
        },
        'importance': {
            'random_forest': rf_importance
        },
        'evaluation': {
            'random_forest': {
                'report': rf_report,
                'cv_scores': cv_scores_rf,
                'confusion_matrix': confusion_matrix(y_test, rf_pred)
            }
        },
        'category_profiles': category_profiles
    }

In [ ]:

df.drop(columns=['Unnamed: 0'])
# Run the analysis
results = analyze_ad_preferences_randomforest(df, target_col='user_pref')

# Visualize the results
# plot_results(results)

print("Random Forest Performance:")
print(results['evaluation']['random_forest']['report'])

# Access category profiles
# print("\nCategory Profiles:")
# print(results['category_profiles'])

Fitting 5 folds for each of 324 candidates, totalling 1620 fits


/usr/local/lib/python3.10/dist-packages/numpy/ma/core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_divisi

Best parameters: {'max_depth': 2, 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 30}
Best cross-validation score: 0.400765393876849
Random Forest Performance:
{1: {'precision': 0.38095238095238093, 'recall': 0.9696969696969697, 'f1-score': 0.5470085470085471, 'support': 33.0}, 3: {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 19.0}, 2: {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 33.0}, 'accuracy': 0.3764705882352941, 'macro avg': {'precision': 0.12698412698412698, 'recall': 0.32323232323232326, 'f1-score': 0.18233618233618235, 'support': 85.0}, 'weighted avg': {'precision': 0.14789915966386555, 'recall': 0.3764705882352941, 'f1-score': 0.21236802413273007, 'support': 85.0}}


# 3.XGBoost

In [ ]:
def analyze_ad_preferences_XGBoost(df, target_col='user_pref', test_size=0.1, random_state=42):
    # Preprocess data
   X = df.copy()
   X = X.drop(columns=['user_pref'])
   y=df[target_col]
   scaler = StandardScaler()
   X = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
   X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
   param_grid = {
      'max_depth':[2,3,4,5,6,7],
      'n_estimators':[1,10,20,30,40,50,60],
      'gamma': [0.1, 0.5, 1, 5],
      'reg_lambda':[ 1,5, 10],
    'alpha': [0, 0.1, 1,5, 10] }
   xgb = XGBClassifier(
    objective='multi:softmax',
    random_state=random_state,
    learning_rate=0.1)
   grid_search = GridSearchCV(
        xgb, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1
    )
   grid_search.fit(X_train, y_train)
   best_xgb = grid_search.best_estimator_
   best_xgb.fit(X_train, y_train)
   print(grid_search.best_params_ )
   best_xgb.fit(X_train, y_train)
   xgb_importance = pd.DataFrame({
        'feature': X.columns,
        'importance': best_xgb.feature_importances_
    }).sort_values('importance', ascending=False)
   xgb_pred = best_xgb.predict(X_test)
   target_names = y_test.unique()
   xgb_report = classification_report(y_test, xgb_pred,
                                      target_names=target_names,
                                      output_dict=True)
   xgb_cm = confusion_matrix(y_test, xgb_pred)
   xgb_cv_scores = cross_val_score(best_xgb, X_train, y_train, cv=5)

   return {
        'models': {
            'xgboost': best_xgb
        },
        'importance': {
            'xgboost': xgb_importance
        },
        'evaluation': {
            'xgboost': {
                'report': xgb_report,
                'cv_scores': xgb_cv_scores,
                'confusion_matrix': xgb_cm
            }
        }
    }



In [ ]:
df.drop(columns=['Unnamed: 0'])
df['user_pref'] -=1

In [ ]:
results = analyze_ad_preferences_XGBoost(df, 'user_pref')


Fitting 5 folds for each of 2520 candidates, totalling 12600 fits


/usr/local/lib/python3.10/dist-packages/joblib/externals/loky/backend/fork_exec.py:38: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid = os.fork()


{'alpha': 5, 'gamma': 0.1, 'max_depth': 2, 'n_estimators': 1, 'reg_lambda': 10}


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
print("XGBoost Performance:")
print(results['evaluation']['xgboost']['report'])


XGBoost Performance:
{0: {'precision': 0.3815789473684211, 'recall': 0.8787878787878788, 'f1-score': 0.5321100917431193, 'support': 33.0}, 2: {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 19.0}, 1: {'precision': 0.4444444444444444, 'recall': 0.12121212121212122, 'f1-score': 0.19047619047619047, 'support': 33.0}, 'accuracy': 0.38823529411764707, 'macro avg': {'precision': 0.2753411306042885, 'recall': 0.3333333333333333, 'f1-score': 0.24086209407310324, 'support': 85.0}, 'weighted avg': {'precision': 0.3206914344685243, 'recall': 0.38823529411764707, 'f1-score': 0.28053349780279085, 'support': 85.0}}


# 5.ANN+LOGISTIC

In [ ]:
def build_hybrid_model(wide_dim, deep_dim, num_classes, hidden_units=32):
    # Wide (Linear) input
    wide_input = Input(shape=(wide_dim,), name='wide_input')

    # Deep (ANN) input
    deep_input = Input(shape=(deep_dim,), name='deep_input')

    # Deep path - simple ANN with one hidden layer
    deep = Dense(hidden_units, activation='relu')(deep_input)

    # Combine wide and deep paths
    combined = Concatenate()([wide_input, deep])

    # Output layer
    if num_classes == 2:
        output = Dense(1, activation='sigmoid')(combined)
    else:
        output = Dense(num_classes+1, activation='softmax')(combined)

    # Create model
    model = Model(inputs=[wide_input, deep_input], outputs=output)

    # Compile model
    if num_classes == 2:
        model.compile(optimizer='adam',
                     loss='binary_crossentropy',
                     metrics=['accuracy'])
    else:
        model.compile(optimizer='adam',
                     loss='sparse_categorical_crossentropy',
                     metrics=['accuracy'])

    return model

def prepare_data(wide_features, deep_features, labels, test_size=0.2):
    # Scale features
    wide_scaler = StandardScaler()
    deep_scaler = StandardScaler()

    wide_scaled = wide_scaler.fit_transform(wide_features)
    deep_scaled = deep_scaler.fit_transform(deep_features)

    # Split data
    (wide_train, wide_test,
     deep_train, deep_test,
     y_train, y_test) = train_test_split(wide_scaled, deep_scaled, labels,
                                        test_size=test_size,
                                        random_state=42)

    return (wide_train, wide_test, deep_train, deep_test,
            y_train, y_test, wide_scaler, deep_scaler)

def train_hybrid_model(wide_features, deep_features, labels,
                      hidden_units=32, epochs=50, batch_size=32,
                      test_size=0.2, verbose=1):
    # Prepare data
    (wide_train, wide_test, deep_train, deep_test,
     y_train, y_test, wide_scaler, deep_scaler) = prepare_data(
        wide_features, deep_features, labels, test_size
    )

    # Get number of classes
    num_classes = len(np.unique(labels)) if len(labels.shape) == 1 else labels.shape[1]

    # Build model
    model = build_hybrid_model(
        wide_dim=wide_features.shape[1],
        deep_dim=deep_features.shape[1],
        num_classes=num_classes,
        hidden_units=hidden_units
    )

    # Train model
    history = model.fit(
        [wide_train, deep_train], y_train,
        validation_data=([wide_test, deep_test], y_test),
        epochs=epochs,
        batch_size=batch_size,
        verbose=verbose
    )

    return model, history, (wide_scaler, deep_scaler)

def predict_hybrid(model, wide_features, deep_features, scalers):

    wide_scaler, deep_scaler = scalers
    wide_scaled = wide_scaler.transform(wide_features)
    deep_scaled = deep_scaler.transform(deep_features)

    return model.predict([wide_scaled, deep_scaled])

In [ ]:
from sklearn.utils import resample

# Concatenate the original DataFrame to the minority classes after duplicating samples
df_minority3 = df[df['user_pref'] == 3]
df_minority2 = df[df['user_pref'] == 2]
df_majority = df[df['user_pref'] == 1]  # Change to majority class name

# Upsample minority class
df_minority_upsampled3 = resample(df_minority3,
                                  replace=True,
                                  n_samples=len(df_majority),
                                  random_state=42)
df_minority_upsampled2= resample(df_minority2,
                                  replace=True,
                                  n_samples=len(df_majority),
                                  random_state=42)

# Combine majority class with upsampled minority class
df_upsampled = pd.concat([df_majority, df_minority_upsampled3,df_minority_upsampled2])

# Check new class distribution
print(df_upsampled['user_pref'].value_counts())

In [ ]:
wide_features = df[['age', 'income', 'degree','full_employ_5_6_7','siting', 'male', 'single', 'peak']]
deep_features = df[[ 'totall_ travel_ time','adv','age','adv_place(0=no,1=adv_in,2=adv_out,3=both)','trip_purpose','number _trips','transfer']]
labels = df['user_pref']

In [ ]:
model, history, scalers = train_hybrid_model(wide_features, deep_features, labels)

Epoch 1/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.3285 - loss: 1.4316 - val_accuracy: 0.3059 - val_loss: 1.3805
Epoch 2/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.3313 - loss: 1.3767 - val_accuracy: 0.3000 - val_loss: 1.3218
Epoch 3/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.3534 - loss: 1.3083 - val_accuracy: 0.3118 - val_loss: 1.2780
Epoch 4/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3517 - loss: 1.2592 - val_accuracy: 0.3000 - val_loss: 1.2472
Epoch 5/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.3842 - loss: 1.2211 - val_accuracy: 0.3000 - val_loss: 1.2203
Epoch 6/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3485 - loss: 1.1962 - val_accuracy: 0.2824 - val_loss: 1.2013
Epoch 7/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4026 - loss: 1.1694 - val_accuracy: 0.3118 - val_loss: 1.1831
Epoch 8/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4035 - loss: 1.1550 - val_accuracy: 0.3118 - val_loss

In [ ]:
def build_hybrid_model(wide_dim, deep_dim, num_classes, hidden_units=32):
    # Wide (Linear) input
    wide_input = Input(shape=(wide_dim,), name='wide_input')

    # Deep (ANN) input
    deep_input = Input(shape=(deep_dim,), name='deep_input')

    # Deep path - simple ANN with one hidden layer
    deep = Dense(hidden_units, activation='relu')(deep_input)

    # Combine wide and deep paths
    combined = Concatenate()([wide_input, deep])

    # Output layer
    if num_classes == 2:
        output = Dense(1, activation='sigmoid')(combined)
    else:
        output = Dense(num_classes+1, activation='softmax')(combined)

    # Create model
    model = Model(inputs=[wide_input, deep_input], outputs=output)

    # Compile model
    if num_classes == 2:
        model.compile(optimizer='adam',
                     loss='binary_crossentropy',
                     metrics=['accuracy'])
    else:
        model.compile(optimizer='adam',
                     loss='sparse_categorical_crossentropy',
                     metrics=['accuracy'])

    return model

def prepare_data(wide_features, deep_features, labels, test_size=0.1):
    # Scale features
    wide_scaler = StandardScaler()
    deep_scaler = StandardScaler()

    wide_scaled = wide_scaler.fit_transform(wide_features)
    deep_scaled = deep_scaler.fit_transform(deep_features)

    # Split data
    (wide_train, wide_test,
     deep_train, deep_test,
     y_train, y_test) = train_test_split(wide_scaled, deep_scaled, labels,
                                        test_size=test_size,
                                        random_state=42)

    return (wide_train, wide_test, deep_train, deep_test,
            y_train, y_test, wide_scaler, deep_scaler)

def train_hybrid_model_with_gridsearch(wide_features, deep_features, labels,
                                     param_grid=None, test_size=0.1, verbose=1):

    # Default parameter grid if none provided
    if param_grid is None:
        param_grid = {
            'hidden_units': [16, 32, 64],
            'epochs': [30, 50,70, 100],
            'batch_size': [16, 32, 64]
        }

    # Prepare data
    (wide_train, wide_val, deep_train, deep_val,
     y_train, y_val, wide_scaler, deep_scaler) = prepare_data(
        wide_features, deep_features, labels, test_size
    )

    # Get number of classes
    num_classes = len(np.unique(labels)) if len(labels.shape) == 1 else labels.shape[1]

    # Initialize tracking variables
    best_score = -np.inf
    best_params = None
    best_model = None
    best_history = None
    results = []

    # Generate parameter combinations
    param_combinations = list(ParameterGrid(param_grid))

    if verbose:
        print(f"Testing {len(param_combinations)} parameter combinations...")

    # Try each parameter combination
    for params in param_combinations:
        if verbose:
            print(f"\nTesting parameters: {params}")

        # Build model with current parameters
        model = build_hybrid_model(
            wide_dim=wide_features.shape[1],
            deep_dim=deep_features.shape[1],
            num_classes=num_classes,
            hidden_units=params['hidden_units']
        )

        # Train model
        history = model.fit(
            [wide_train, deep_train], y_train,
            validation_data=([wide_val, deep_val], y_val),
            epochs=params['epochs'],
            batch_size=params['batch_size'],
            verbose=0 if verbose==1 else verbose
        )

        # Get validation score
        val_score = max(history.history['val_accuracy'])

        # Store results
        results.append({
            'params': params,
            'val_score': val_score
        })

        if verbose:
            print(f"Validation accuracy: {val_score:.4f}")

        # Update best parameters if necessary
        if val_score > best_score:
            best_score = val_score
            best_params = params
            best_model = model
            best_history = history

    if verbose:
        print("\nGrid search completed.")
        print(f"Best parameters: {best_params}")
        print(f"Best validation accuracy: {best_score:.4f}")

    return best_model, best_history, (wide_scaler, deep_scaler), {
        'best_params': best_params,
        'best_score': best_score,
        'results': results
    }

def predict_hybrid(model, wide_features, deep_features, scalers):
    wide_scaler, deep_scaler = scalers
    wide_scaled = wide_scaler.transform(wide_features)
    deep_scaled = deep_scaler.transform(deep_features)

    return model.predict([wide_scaled, deep_scaled])



In [ ]:
from sklearn.utils import resample

# Concatenate the original DataFrame to the minority classes after duplicating samples
df_minority3 = df[df['user_pref'] == 3]
df_minority2 = df[df['user_pref'] == 2]
df_majority = df[df['user_pref'] == 1]  # Change to majority class name

# Upsample minority class
df_minority_upsampled3 = resample(df_minority3,
                                  replace=True,
                                  n_samples=len(df_majority),
                                  random_state=42)
df_minority_upsampled2= resample(df_minority2,
                                  replace=True,
                                  n_samples=len(df_majority),
                                  random_state=42)

# Combine majority class with upsampled minority class
df_upsampled = pd.concat([df_majority, df_minority_upsampled3,df_minority_upsampled2])

# Check new class distribution
print(df_upsampled['user_pref'].value_counts())

user_pref
1    344
3    344
2    344
Name: count, dtype: int64


In [ ]:
wide_features = df_upsampled[['age', 'income', 'full_employ_5_6_7','siting', 'male', 'peak']]
deep_features = df_upsampled[[ 'adv','age','adv_place(0=no,1=adv_in,2=adv_out,3=both)','trip_purpose','number _trips']]
labels = df_upsampled['user_pref']

In [ ]:
model, history, scalers, search_results = train_hybrid_model_with_gridsearch(
    wide_features,
    deep_features,
    labels
)

Testing 36 parameter combinations...

Testing parameters: {'batch_size': 16, 'epochs': 30, 'hidden_units': 16}
Validation accuracy: 0.4135

Testing parameters: {'batch_size': 16, 'epochs': 30, 'hidden_units': 32}
Validation accuracy: 0.4327

Testing parameters: {'batch_size': 16, 'epochs': 30, 'hidden_units': 64}
Validation accuracy: 0.4038

Testing parameters: {'batch_size': 16, 'epochs': 50, 'hidden_units': 16}
Validation accuracy: 0.3750

Testing parameters: {'batch_size': 16, 'epochs': 50, 'hidden_units': 32}
Validation accuracy: 0.3942

Testing parameters: {'batch_size': 16, 'epochs': 50, 'hidden_units': 64}
Validation accuracy: 0.4712

Testing parameters: {'batch_size': 16, 'epochs': 70, 'hidden_units': 16}
Validation accuracy: 0.3846

Testing parameters: {'batch_size': 16, 'epochs': 70, 'hidden_units': 32}
Validation accuracy: 0.3654

Testing parameters: {'batch_size': 16, 'epochs': 70, 'hidden_units': 64}
Validation accuracy: 0.3942

Testing parameters: {'batch_size': 16, 'epoc

# 6.Logistic

In [ ]:
def prepare_data(features, target, test_size=0.2, random_state=42):
    # Scale features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(features)

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, target,
        test_size=test_size,
        random_state=random_state,
        stratify=target
    )

    return X_train, X_test, y_train, y_test, scaler

def train_logistic_model(X_train, y_train, multi_class='auto', max_iter=1000):
    model = LogisticRegression(max_iter=max_iter, random_state=42)
    model.fit(X_train, y_train)
    return model

def analyze_feature_importance(model, feature_names):
    # Get coefficients
    if len(model.classes_) == 2:
        coefficients = model.coef_[0]
        classes = ['Class 0', 'Class 1']
    else:
        coefficients = model.coef_
        classes = [f'Class {i}' for i in range(len(model.classes_))]

    # Create feature importance DataFrame
    importance_df = pd.DataFrame()

    for idx, class_name in enumerate(classes):
        if len(model.classes_) == 2:
            coef = coefficients
        else:
            coef = coefficients[idx]

        class_importance = pd.DataFrame({
            'Feature': feature_names,
            'Importance': np.abs(coef),
            'Effect': coef,
            'Class': class_name
        })
        importance_df = pd.concat([importance_df, class_importance])

    return importance_df

def plot_feature_importance(importance_df, top_n=10):
    plt.figure(figsize=(12, 6))

    # Get top features by average absolute importance
    top_features = (importance_df.groupby('Feature')['Importance']
                   .mean()
                   .sort_values(ascending=False)
                   .head(top_n)
                   .index)

    # Filter data for top features
    plot_data = importance_df[importance_df['Feature'].isin(top_features)]

    # Create bar plot
    sns.barplot(data=plot_data,
                x='Effect',
                y='Feature',
                hue='Class',
                dodge=True)

    plt.title(f'Top {top_n} Most Important Features by Class')
    plt.xlabel('Coefficient Value (Effect on Prediction)')
    plt.tight_layout()
    plt.show()

def evaluate_model(model, X_test, y_test):
    # Make predictions
    y_pred = model.predict(X_test)

    # Calculate metrics
    report = classification_report(y_test, y_pred, output_dict=True ,zero_division=0)
    conf_matrix = confusion_matrix(y_test, y_pred)

    # Plot confusion matrix
    # plt.figure(figsize=(8, 6))
    # sns.heatmap(conf_matrix,
    #             annot=True,
    #             fmt='d',
    #             xticklabels=model.classes_,
    #             yticklabels=model.classes_)
    # plt.title('Confusion Matrix')
    # plt.ylabel('True Label')
    # plt.xlabel('Predicted Label')
    # plt.tight_layout()
    # plt.show()

    return report

def predict_new_data(model, features, scaler):
    # Scale features
    X_scaled = scaler.transform(features)

    # Get predictions and probabilities
    predictions = model.predict(X_scaled)
    probabilities = model.predict_proba(X_scaled)

    return predictions, probabilities

def run_analysis(features, target):
    # Prepare data
    X_train, X_test, y_train, y_test, scaler = prepare_data(features, target)

    # Train model
    model = train_logistic_model(X_train, y_train)

    # Analyze feature importance
    importance_df = analyze_feature_importance(model, features.columns)

    # Plot feature importance
    # plot_feature_importance(importance_df)

    # Evaluate model
    evaluation = evaluate_model(model, X_test, y_test)

    return model, scaler, importance_df, evaluation

In [ ]:
features = df[['age', 'income', 'degree','full_employ_5_6_7','siting', 'male', 'single',
               'peak','totall_ travel_ time','adv','age']]
target =df['user_pref']

model, scaler, importance_df, evaluation = run_analysis(features, target)

In [ ]:
print(evaluation )

{'1': {'precision': 0.3684210526315789, 'recall': 0.6086956521739131, 'f1-score': 0.45901639344262296, 'support': 69.0}, '2': {'precision': 0.4, 'recall': 0.045454545454545456, 'f1-score': 0.08163265306122448, 'support': 44.0}, '3': {'precision': 0.35294117647058826, 'recall': 0.3157894736842105, 'f1-score': 0.3333333333333333, 'support': 57.0}, 'accuracy': 0.36470588235294116, 'macro avg': {'precision': 0.3737874097007224, 'recall': 0.32331322377088967, 'f1-score': 0.29132745994572695, 'support': 170.0}, 'weighted avg': {'precision': 0.37140411582589694, 'recall': 0.36470588235294116, 'f1-score': 0.31919981107196976, 'support': 170.0}}
